# SmartHandover — Day 2: GoEmotions Zero-Shot Baseline

Corre o modelo `j-hartmann/emotion-english-distilroberta-base` sobre o MELD, mapeia para as 5 classes-alvo e compara com o VADER.

**Toggle:** muda `USE_MOCK = True` para dados dummy.

In [ ]:
#%pip install transformers vaderSentiment pandas scikit-learn tqdm matplotlib datasets librosa soundfile torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
# === CONFIG ===
USE_MOCK = False   # True = dados dummy, False = MELD real
DEVICE = 0         # -1 = CPU, 0 = GPU
BATCH_SIZE = 64

In [3]:
import os, sys, time
import pandas as pd
from tqdm.notebook import tqdm

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

In [4]:
from src.classifiers.goemo_classifier import GoEmotionsClassifier, GOEMO_LABELS

TARGET_LABELS = ["anger", "frustration", "sadness", "neutral", "satisfaction"]
TARGET_LABEL2ID = {label: i for i, label in enumerate(TARGET_LABELS)}

## 1. Carregar dados

In [5]:
def create_mock_data():
    samples = [
        ("I am so angry at you right now!", "anger"),
        ("This is absolutely unacceptable, I'm furious!", "anger"),
        ("You ruined everything, I hate this!", "anger"),
        ("This is so frustrating, nothing works.", "frustration"),
        ("I've been waiting for an hour, this is ridiculous.", "frustration"),
        ("Why does this keep happening to me?", "frustration"),
        ("I feel so sad and lonely today.", "sadness"),
        ("It breaks my heart to see this.", "sadness"),
        ("Nothing seems to matter anymore.", "sadness"),
        ("The meeting is at 3pm.", "neutral"),
        ("Sure, that works for me.", "neutral"),
        ("Let me know when you're ready.", "neutral"),
        ("Okay.", "neutral"),
        ("This is wonderful, I'm so happy!", "satisfaction"),
        ("Great job, I really appreciate your help!", "satisfaction"),
        ("Everything worked out perfectly!", "satisfaction"),
    ]
    return pd.DataFrame(samples, columns=["text", "target_emotion"])


def load_meld_as_dataframe():
    from src.data.load_meld import load_meld
    rows = []
    for split in ["train", "validation", "test"]:
        print(f"  Loading MELD split: {split} ...")
        ds = load_meld(split=split, streaming=False)
        for example in ds:
            rows.append({"text": example["text"], "target_emotion": example["target_emotion"], "split": split})
    return pd.DataFrame(rows)


if USE_MOCK:
    print("[INFO] A usar dados MOCK.")
    df = create_mock_data()
else:
    print("[INFO] A carregar MELD real...")
    try:
        df = load_meld_as_dataframe()
    except Exception as e:
        print(f"[WARN] Falha: {e}\n[WARN] Fallback para mock.")
        df = create_mock_data()

print(f"\nTotal amostras: {len(df)}")
df["target_emotion"].value_counts()

[INFO] A carregar MELD real...
  Loading MELD split: train ...
  Loading MELD split: validation ...
  Loading MELD split: test ...

Total amostras: 12070


target_emotion
neutral         6434
satisfaction    2308
anger           1968
sadness         1002
frustration      358
Name: count, dtype: int64

## 2. Carregar modelo e correr inferencia em batch

In [6]:
print(f"Loading GoEmotions model (device={DEVICE})...")
t0 = time.time()
classifier = GoEmotionsClassifier(device=DEVICE, batch_size=BATCH_SIZE)
print(f"Modelo carregado em {time.time() - t0:.1f}s")

texts = df["text"].tolist()
true_labels = df["target_emotion"].tolist()

print(f"\nA correr inferencia em batch (batch_size={BATCH_SIZE})...")
all_probs = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="GoEmotions"):
    batch = texts[i : i + BATCH_SIZE]
    all_probs.extend(classifier.predict_batch(batch))

# Construir DataFrame de resultados
rows = []
for idx, probs in enumerate(all_probs):
    row = {"text": texts[idx], "true_label": true_labels[idx], "predicted_class": classifier.map_to_target_class(probs)}
    for label in GOEMO_LABELS:
        row[f"goemo_{label}"] = probs.get(label, 0.0)
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.head(10)

Loading GoEmotions model (device=0)...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo carregado em 1.4s

A correr inferencia em batch (batch_size=64)...


GoEmotions:   0%|          | 0/189 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,text,true_label,predicted_class,goemo_anger,goemo_disgust,goemo_fear,goemo_joy,goemo_neutral,goemo_sadness,goemo_surprise
0,also I was the point person on my companys tr...,neutral,neutral,0.011712,0.009903,0.023333,0.020589,0.841257,0.031842,0.061363
1,You mustve had your hands full.,neutral,neutral,0.211492,0.145589,0.071229,0.028871,0.389689,0.066029,0.087101
2,No dont I beg of you!,frustration,frustration,0.329371,0.013111,0.605006,0.007137,0.007027,0.022894,0.015454
3,"All right then, well have a definite answer f...",neutral,neutral,0.021566,0.018279,0.028187,0.246362,0.666053,0.012839,0.006713
4,Absolutely. You can relax,neutral,neutral,0.007463,0.010125,0.004432,0.344746,0.578668,0.048266,0.006300
5,That I did. That I did.,neutral,neutral,0.072654,0.029890,0.005867,0.014911,0.852306,0.018975,0.005397
6,So lets talk a little bit about your duties.,neutral,neutral,0.028570,0.060439,0.017375,0.031890,0.840250,0.018048,0.003429
7,"Now youll be heading a whole division, so you...",neutral,neutral,0.038508,0.037879,0.007674,0.010714,0.878905,0.009966,0.016353
8,I see.,neutral,neutral,0.035709,0.086793,0.006965,0.020525,0.168957,0.004740,0.676312
9,But therell be perhaps 30 people under you so...,neutral,neutral,0.013564,0.017208,0.004985,0.010157,0.924977,0.010461,0.018649


## 3. Guardar CSV

In [7]:
output_path = os.path.join("..", "data", "processed", "goemo_predictions.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
results_df.to_csv(output_path, index=False)
print(f"Guardado em: {output_path}")

Guardado em: ..\data\processed\goemo_predictions.csv


## 4. Metricas

In [8]:
from src.evaluation.metrics import compute_metrics, print_metrics

y_true = [TARGET_LABEL2ID[l] for l in results_df["true_label"]]
y_pred = [TARGET_LABEL2ID[l] for l in results_df["predicted_class"]]

metrics = compute_metrics(y_true, y_pred, target_names=TARGET_LABELS)
print_metrics(metrics)


  Accuracy     : 0.5476
  Weighted F1  : 0.5596
  Macro F1     : 0.4049
  Frustration R: 0.4274  << KEY METRIC

  Class             Prec    Rec     F1  Support
  -------------------------------------------
  anger            0.436  0.372  0.401     1968
  frustration      0.088  0.427  0.146      358 <<
  sadness          0.456  0.221  0.297     1002
  neutral          0.689  0.714  0.701     6434
  satisfaction     0.606  0.396  0.479     2308

  Confusion Matrix (rows=true, cols=pred):
            anger  frust  sadne  neutr  satis
    anger     732    414     39    679    104
    frust      59    153     11    125     10
    sadne     138    219    221    383     41
    neutr     448    790    167   4591    438
    satis     303    156     47    889    913


## 5. Comparacao VADER vs GoEmotions

In [11]:
vader_path = os.path.join("..", "data", "processed", "vader_predictions.csv")
if os.path.exists(vader_path):
    vader_df = pd.read_csv(vader_path)
    v_true = [TARGET_LABEL2ID[l] for l in vader_df["true_label"]]
    v_pred = [TARGET_LABEL2ID[l] for l in vader_df["predicted_class"]]
    vader_metrics = compute_metrics(v_true, v_pred, target_names=TARGET_LABELS)

    comparison = pd.DataFrame({
        "Metric": ["Accuracy", "Weighted F1", "Macro F1", "Frustration Recall"],
        "VADER (Day 1)": [vader_metrics["accuracy"], vader_metrics["weighted_f1"], vader_metrics["macro_f1"], vader_metrics["frustration_recall"]],
        "GoEmotions (Day 2)": [metrics["accuracy"], metrics["weighted_f1"], metrics["macro_f1"], metrics["frustration_recall"]],
    })
    comparison["Delta"] = comparison["GoEmotions (Day 2)"] - comparison["VADER (Day 1)"]
    comparison.style.format({"VADER (Day 1)": "{:.4f}", "GoEmotions (Day 2)": "{:.4f}", "Delta": "{:+.4f}"})
else:
    print("vader_predictions.csv nao encontrado — corre o Day 1 primeiro.")